<a href="https://colab.research.google.com/github/1Bur1/clothes-image-Classifier/blob/main/02_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 - Evaluate the Model

**Rule:** The test set is touched **exactly once** - right here, at the very end.  
Using it earlier would give a falsely optimistic accuracy.

---

## Notebook Structure
1. Setup & Imports
2. `FashionEvaluator` Class Definition
3. Load Model & Data
4. REQUIRED - Test Accuracy (model.evaluate())
5. REQUIRED - Classification Report
6. NOT REQUIRED - Class Names in Report
7. REQUIRED - Confusion Matrix (10x10)
8. NOT REQUIRED - Class Name Labels on Matrix Axes
9. REQUIRED - Error Analysis (9 Misclassified Images)
10. NOT REQUIRED - Save Misclassified Grid to reports/
11. NOT REQUIRED - Summary Block

## 1. Setup & Imports

| Library | Purpose |
|---|---|
| `numpy` | Array operations |
| `matplotlib` | Plotting confusion matrix and error grid |
| `tensorflow / keras` | Loading the saved model |
| `sklearn.metrics` | `classification_report` and `ConfusionMatrixDisplay` |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

os.makedirs('../reports', exist_ok=True)
print(f'TensorFlow version : {tf.__version__}')
print('Ready.')


## 2. `FashionEvaluator` Class Definition

All evaluation logic lives in one class - same pattern as `FashionClassifier` from Phase 2.

| Method | What it does |
|---|---|
| `load_data()` | Reloads test set with same preprocessing as training |
| `load_model()` | Loads the saved .h5 model from Phase 2 |
| `predict()` | np.argmax(model.predict(X_test), axis=1) |
| `evaluate()` | model.evaluate() - official test accuracy |
| `show_classification_report()` | Per-class precision, recall, F1 |
| `plot_confusion_matrix()` | 10x10 heatmap saved to reports/ |
| `error_analysis()` | 9 misclassified images grid |

In [ ]:
class FashionEvaluator:

    CLASS_NAMES = [
        'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat','Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

    def __init__(self):
        self.X_test = None
        self.y_test = None
        self.model  = None
        self.y_pred = None

    def load_data(self):
        _, (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
        self.X_test = (X_test / 255.0).reshape(-1, 28, 28, 1)
        self.y_test = y_test
        print(f'Test set loaded: {self.X_test.shape}  ({len(self.X_test):,} images)')
        return self

    def load_model(self, path='../models/fashion_cnn.h5'):
        self.model = keras.models.load_model(path)
        print(f'Model loaded from: {path}')
        self.model.summary()
        return self

    def predict(self):
        self.y_pred = np.argmax(self.model.predict(self.X_test), axis=1)
        print(f'Predictions generated for {len(self.y_pred):,} images.')
        return self

    def evaluate(self):
        loss, accuracy = self.model.evaluate(self.X_test, self.y_test, verbose=0)
        print(f'Test Loss     : {loss:.4f}')
        print(f'Test Accuracy : {accuracy:.4f}  ({accuracy*100:.2f}%)')
        return self

    def show_classification_report(self, use_class_names=False):
        names = self.CLASS_NAMES if use_class_names else None
        print(classification_report(self.y_test, self.y_pred, target_names=names))
        return self

    def plot_confusion_matrix(self, use_class_names=False,
                               save_path='../reports/confusion_matrix.png'):
        fig, ax = plt.subplots(figsize=(12, 10))
        labels   = self.CLASS_NAMES if use_class_names else None
        rotation = 45 if use_class_names else 0
        ConfusionMatrixDisplay.from_predictions(
            self.y_test, self.y_pred,
            display_labels=labels,
            xticks_rotation=rotation,
            colorbar=True, ax=ax
        )
        ax.set_title('Confusion Matrix - Fashion-MNIST Test Set', fontsize=14)
        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f'Saved -> {save_path}')
        return self

    def error_analysis(self, n=9, save_path=None):
        wrong_idx = np.where(self.y_pred != self.y_test)[0]
        print(f'Total misclassified: {len(wrong_idx):,} / {len(self.y_test):,} '
              f'({len(wrong_idx)/len(self.y_test)*100:.1f}% error rate)')
        sample_idx = wrong_idx[:n]
        cols = 3
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(10, rows * 3.5))
        fig.suptitle(f'Error Analysis - {n} Misclassified Images', fontsize=14)
        for i, ax in enumerate(axes.flat):
            if i < len(sample_idx):
                idx = sample_idx[i]
                ax.imshow(self.X_test[idx].squeeze(), cmap='gray')
                true_label = self.CLASS_NAMES[self.y_test[idx]]
                pred_label = self.CLASS_NAMES[self.y_pred[idx]]
                ax.set_title(f'True : {true_label}\nPred : {pred_label}',
                             fontsize=9, color='red')
            ax.axis('off')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150)
            print(f'Saved -> {save_path}')
        plt.show()
        return self


print('FashionEvaluator class defined.')


## 3. Load Model & Data

Load the model saved at the end of Phase 2 and reload the test set.

In [ ]:
ev = FashionEvaluator()
ev.load_data()
ev.load_model('../models/fashion_cnn.h5')
ev.predict()


## 4.  - Test Accuracy

Compute official test accuracy using `model.evaluate()` as required by the project brief.

In [ ]:
# REQUIRED
ev.evaluate()


## 5.- Classification Report

Print per-class precision, recall, and F1 as required by the project brief.

In [ ]:
# REQUIRED - skeleton from PDF
ev.show_classification_report(use_class_names=False)


---
## 6. NOT REQUIRED - Added to improve the project
### Class Names in Classification Report

Same report but with `target_names=CLASS_NAMES`.  
Replaces raw numbers (0, 1, 2...) with readable labels (T-shirt, Trouser, Pullover...).  
**This cell is not required by the project brief.**

---

In [ ]:
# NOT REQUIRED - added to improve the project
ev.show_classification_report(use_class_names=True)


## 7. - Confusion Matrix (10x10)

Plot and save the 10x10 confusion matrix to `reports/` as required by the project brief.

In [ ]:
# REQUIRED - skeleton from PDF
ev.plot_confusion_matrix(use_class_names=False,
                          save_path='../reports/confusion_matrix.png')


---
## 8. NOT REQUIRED - Added to improve the project
### Class Name Labels on Confusion Matrix Axes

Same matrix but both axes show class names instead of numbers 0-9.  
Uses `display_labels=CLASS_NAMES` and `xticks_rotation=45`.  
**This cell is not required by the project brief.**

---

In [ ]:
# NOT REQUIRED - added to improve the project
ev.plot_confusion_matrix(use_class_names=True,save_path='../reports/confusion_matrix_named.png')


## 9. - Error Analysis (9 Misclassified Images)

Show 9 misclassified images with their true and predicted labels, as required by the project brief.

In [ ]:
# REQUIRED - 9 misclassified images with true and predicted labels
ev.error_analysis(n=9, save_path=None)


---
## 10. NOT REQUIRED - Added to improve the project fairness
### Save Misclassified Grid to reports/

Same error analysis grid but saved to `reports/misclassified_samples.png`.  
Keeps all visuals together for the final report.  
**This cell is not required by the project brief.**

---

In [ ]:
# NOT REQUIRED - added to improve the project
ev.error_analysis(n=9, save_path='../reports/misclassified_samples.png')


---
## 11. NOT REQUIRED - Added to improve the project
### Summary Block

A clean summary of all evaluation numbers in one place.  

**This cell is not required by the project brief.**

---

In [ ]:
# NOT REQUIRED - added to improve the project
wrong    = np.sum(ev.y_pred != ev.y_test)
total    = len(ev.y_test)
accuracy = np.mean(ev.y_pred == ev.y_test)

print('=' * 50)
print('  PHASE 3 EVALUATION COMPLETE')
print('=' * 50)
print(f'  Total test images   : {total:,}')
print(f'  Correct predictions : {total - wrong:,}')
print(f'  Wrong predictions   : {wrong:,}')
print(f'  Test Accuracy       : {accuracy*100:.2f}%')
print('=' * 50)
print('Files saved to reports/:')
print('  confusion_matrix.png')
print('  confusion_matrix_named.png')
print('  misclassified_samples.png')
